# Markov Random Fields

**Companion lesson:** https://ml-viz.vercel.app/courses/graphical-models/02-markov-random-fields

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A 2-node Ising MRF — potentials and the partition function

Two spins that prefer to align: $\psi(x_1,x_2)=e^{\beta x_1 x_2}$. We compute the distribution explicitly, including the partition function $Z$.

In [ ]:
beta = 1.0
states = [(+1,+1), (-1,-1), (+1,-1), (-1,+1)]
psi = np.array([np.exp(beta*a*b) for a,b in states])
Z = psi.sum()
for (a,b), p in zip(states, psi/Z):
    print(f'x1={a:+d} x2={b:+d}: P = {p:.3f}')
print('P(aligned) =', round((psi[0]+psi[1])/Z, 3), ' (Z =', round(Z,3), ')')

## Image denoising — an MRF you can see

Each pixel is a spin. Energy = a **data term** (stay close to the noisy observation) + a **smoothness term** (agree with neighbors). We minimize it with Iterated Conditional Modes (ICM): repeatedly set each pixel to its lowest-energy value given its neighbors.

In [ ]:
rng = np.random.RandomState(0)
# clean binary image: a filled square
img = -np.ones((40, 40)); img[10:30, 10:30] = 1
noisy = img.copy()
flip = rng.rand(*img.shape) < 0.15            # 15% salt-and-pepper noise
noisy[flip] *= -1

def denoise(y, eta=2.0, beta=2.0, sweeps=6):
    x = y.copy()
    H, W = x.shape
    for _ in range(sweeps):
        for i in range(H):
            for j in range(W):
                nb = 0
                if i>0: nb += x[i-1,j]
                if i<H-1: nb += x[i+1,j]
                if j>0: nb += x[i,j-1]
                if j<W-1: nb += x[i,j+1]
                # energy(x_ij=+1) vs (-1); pick lower energy
                e_pos = -eta*y[i,j]*1 - beta*nb*1
                e_neg = -eta*y[i,j]*-1 - beta*nb*-1
                x[i,j] = 1 if e_pos < e_neg else -1
    return x

clean = denoise(noisy)
err_before = (noisy != img).mean(); err_after = (clean != img).mean()
print(f'pixel error: noisy {err_before:.3f} -> denoised {err_after:.3f}')
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, t in zip(ax, [img, noisy, clean], ['original','noisy (15%)','MRF denoised']):
    a.imshow(im, cmap='gray'); a.set_title(t); a.axis('off')
plt.show()

## Key takeaways

- MRFs score configurations with **potentials** over cliques; $Z$ normalizes them.
- Writing potentials as $e^{-\text{energy}}$ connects MRFs to physics (the Ising model).
- Image denoising = minimize data + smoothness energy; ICM does it pixel-by-pixel.
- The smoothness prior cleans noise while preserving the underlying shape.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Agreement in the 2-node Ising model

With potential $\psi(x_1, x_2) = e^{J x_1 x_2}$ over spins $\pm 1$, the probability the two spins **agree** has a closed form — sum the two agreeing configurations and divide by the partition function:

$$P(x_1 = x_2) = \frac{2e^J}{2e^J + 2e^{-J}} = \sigma(2J)$$

Implement it by literal enumeration of the 4 states (that's the partition function in miniature).

In [ ]:
def ising_unnorm(x1, x2, J):
    """Unnormalized potential exp(J * x1 * x2)."""
    # TODO(you)
    return ...


def p_agree(J):
    """P(x1 == x2) in the 2-spin Ising model with coupling J."""
    states = [(1, 1), (1, -1), (-1, 1), (-1, -1)]

    # TODO(you): the partition function Z = sum of potentials over all 4 states
    Z = ...

    # TODO(you): sum of potentials over the agreeing states, divided by Z
    return ...

In [ ]:
# Checks — run me
assert abs(p_agree(0.0) - 0.5) < 1e-12, "J = 0: no coupling, agreement is a coin flip"
assert p_agree(2.0) > 0.95, "strong positive coupling -> spins almost always agree"
assert p_agree(-2.0) < 0.05, "negative coupling -> they anti-align"
assert abs(p_agree(1.0) - np.exp(1) / (np.exp(1) + np.exp(-1))) < 1e-12, "closed form: sigmoid(2J)"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def ising_unnorm(x1, x2, J):
    return np.exp(J * x1 * x2)


def p_agree(J):
    states = [(1, 1), (1, -1), (-1, 1), (-1, -1)]
    Z = sum(ising_unnorm(a, b, J) for a, b in states)
    return sum(ising_unnorm(a, b, J) for a, b in states if a == b) / Z
```

</details>

### Exercise 2 — The denoising energy

The 1D version of the image-denoising objective: a smoothness term that rewards neighboring agreement and a data term that rewards matching the noisy observation $\mathbf{y}$:

$$E(\mathbf{x}) = -\beta \sum_i x_i x_{i+1} \; - \; \lambda \sum_i x_i y_i$$

Lower is better. The checks stage the trade-off: with $\beta > \lambda$, smoothing out a lone flipped pixel beats copying the observation — and with $\beta = 0$ copying wins, because nothing rewards smoothness.

In [ ]:
def mrf_energy(x, y, beta, lam):
    """Energy of configuration x given observation y (both arrays of +/-1)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO(you): smoothness term -beta * sum of x_i * x_{i+1} (hint: x[:-1] * x[1:])
    smooth = ...

    # TODO(you): data term -lam * sum of x_i * y_i
    data = ...

    return smooth + data

In [ ]:
# Checks — run me
y = np.array([1, 1, -1, 1, 1])     # observed, with one flipped pixel
x_clean = np.array([1, 1, 1, 1, 1])
x_copy = y.copy()

assert mrf_energy(x_clean, y, beta=1.0, lam=0.5) < mrf_energy(x_copy, y, beta=1.0, lam=0.5), \
    "smoothing beats copying the noisy observation when beta > lam"
assert mrf_energy(x_copy, y, beta=0.0, lam=0.5) < mrf_energy(x_clean, y, beta=0.0, lam=0.5), \
    "no smoothness term -> copying the data is optimal"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def mrf_energy(x, y, beta, lam):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    smooth = -beta * np.sum(x[:-1] * x[1:])
    data = -lam * np.sum(x * y)
    return smooth + data
```

</details>